# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Load data

In [2]:
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,XXXXXXXXX,Tana,19490,Tate,XXXXXXXXX,Home Office,CA,3200 Amber Bend,90027.0,2,Fitness,34.125946,-118.291016,Pacific Asia,Townsville,Australia,19490,1/13/2018 11:45,75937,1360,22.940001,0.07,179252,327.75,0.08,1,327.75,304.809998,22.860001,Oceania,Queensland,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Orli,19489,Hendricks,XXXXXXXXX,Corporate,PR,8671 Iron Anchor Corners,725.0,2,Fitness,18.253769,-66.037048,Pacific Asia,Townsville,Australia,19489,1/13/2018 11:24,75936,1360,29.500000,0.09,179251,327.75,0.45,1,327.75,298.250000,134.210007,Oceania,Queensland,PENDING_PAYMENT,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


# Holdout data split

In [3]:
# Convert date column to datetime
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Sort by date (oldest to most recent)
df = df.sort_values('order date (DateOrders)').reset_index(drop=True)

# Calculate split index
split_index = int(len(df) * 0.90)

# Split
df_working = df.iloc[:split_index].reset_index(drop=True)
holdout_data = df.iloc[split_index:].reset_index(drop=True)

# Verify
print(f"Full dataset:    {df.shape[0]} rows")
print(f"Working set:     {df_working.shape[0]} rows")
print(f"Holdout set:     {holdout_data.shape[0]} rows")
print(f"\nWorking set date range: {df_working['order date (DateOrders)'].min()} → {df_working['order date (DateOrders)'].max()}")
print(f"Holdout set date range: {holdout_data['order date (DateOrders)'].min()} → {holdout_data['order date (DateOrders)'].max()}")

Full dataset:    180519 rows
Working set:     162467 rows
Holdout set:     18052 rows

Working set date range: 2015-01-01 00:00:00 → 2017-08-06 17:56:00
Holdout set date range: 2017-08-06 17:56:00 → 2018-01-31 23:38:00


# Data Cleaning

In [4]:
df_working.isnull().sum()[df.isnull().sum() > 0]

Customer Lname              0
Customer Zipcode            0
Order Zipcode          137627
Product Description    162467
dtype: int64

In [5]:
df_working.describe()

,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Id,Customer Id,Customer Zipcode,Department Id,Latitude,Longitude,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Price,Product Status
count,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467,162467.000000,162467.000000,162467.000000,162467.000000,162467.00000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000,24840.000000,162467.000000,162467.000000,0.0,162467.000000,162467.0
mean,3.498440,2.933636,21.257907,178.090485,0.547871,30.103664,6213.271427,35946.146571,5.346704,29.726495,-84.943787,6213.271427,2016-04-19 08:27:25.290059,32496.055199,661.013492,20.100232,0.101683,81234.00000,132.520939,0.120055,2.189799,198.190476,178.090485,21.257907,55426.132327,661.013492,30.103664,NaN,132.520939,0.0
min,0.000000,0.000000,-1374.859985,7.490000,0.000000,2.000000,1.000000,603.000000,2.000000,-33.937553,-158.025986,1.000000,2015-01-01 00:00:00,1.000000,19.000000,0.000000,0.000000,1.00000,9.990000,-2.750000,1.000000,9.990000,7.490000,-1374.859985,1040.000000,19.000000,2.000000,NaN,9.990000,0.0
25%,2.000000,2.000000,7.300000,104.379997,0.000000,18.000000,3121.000000,725.000000,4.000000,18.265705,-98.469292,3121.000000,2015-08-26 09:39:00,16264.000000,403.000000,5.500000,0.040000,40617.50000,50.000000,0.080000,1.000000,119.980003,104.379997,7.300000,23464.000000,403.000000,18.000000,NaN,50.000000,0.0
50%,3.000000,4.000000,31.480000,161.970001,1.000000,29.000000,6163.000000,19380.000000,5.000000,33.139019,-76.873726,6163.000000,2016-04-19 03:11:00,32481.000000,627.000000,14.000000,0.100000,81234.00000,59.990002,0.270000,1.000000,199.919998,161.970001,31.480000,59405.000000,627.000000,29.000000,NaN,59.990002,0.0
75%,5.000000,4.000000,63.669998,242.500000,1.000000,45.000000,9344.000000,78207.000000,7.000000,39.279617,-66.370583,9344.000000,2016-12-12 04:26:00,48720.000000,1004.000000,28.799999,0.160000,121850.50000,199.990005,0.360000,3.000000,299.950012,242.500000,63.669998,90008.000000,1004.000000,45.000000,NaN,199.990005,0.0
max,6.000000,4.000000,249.979996,499.950012,1.000000,48.000000,12435.000000,99205.000000,7.000000,48.781933,115.263077,12435.000000,2017-08-06 17:56:00,64994.000000,1073.000000,125.000000,0.250000,162467.00000,399.980011,0.500000,5.000000,500.000000,499.950012,249.979996,99301.000000,1073.000000,48.000000,NaN,399.980011,0.0
std,1.624353,1.374159,97.039165,101.199028,0.497705,13.706613,3593.437020,37545.521014,1.479824,9.813931,21.460668,3593.437020,NaN,18759.089450,310.167425,19.615210,0.070425,46900.32743,116.746262,0.467250,1.468200,111.201571,101.199028,97.039165,31919.279101,310.167425,13.706613,NaN,116.746262,0.0


In [6]:
df_working['Late_delivery_risk'].value_counts(normalize=True) * 100

Late_delivery_risk
1    54.787126
0    45.212874
Name: proportion, dtype: float64

In [7]:
df_working.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Email',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Password',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'Product Card Id',
 'Product Category Id',
 'Product Description',
 'Product Image',
 'Product Name',
 'Product P

In [11]:
df_working.shape

(162467, 35)

# Pre-Processing

## Removing unnecessary columns

In [10]:
columns_to_drop = [
    'Customer Email', 'Product Image', 'Product Status', 
    'Order Zipcode', 'Product Description',
    'Days for shipping (real)', 'Delivery Status', 
    'shipping date (DateOrders)', 'Order Status',
    'Benefit per order', 'Order Profit Per Order',
    'Customer Password', 'Customer Fname', 'Customer Lname',
    'Customer Street', 'Order Id', 'Order Item Id', 'Order Customer Id'
]

df_working = df_working.drop(columns=columns_to_drop)
print(f"Remaining columns: {df_working.shape[1]}")
print(df_working.columns.tolist())

Remaining columns: 35
['Type', 'Days for shipment (scheduled)', 'Sales per customer', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Region', 'Order State', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'Shipping Mode']


In [13]:
columns_to_drop_redundant = [
    'Order Item Total',
    'Sales per customer',
    'Order Item Product Price',
    'Order Item Discount',
    'Category Id',
    'Product Category Id',
    'Department Id',
    'Order Item Cardprod Id'
]

df_working = df_working.drop(columns=columns_to_drop_redundant)
print(f"Remaining columns: {df_working.shape[1]}")
print(df_working.columns.tolist())

Remaining columns: 27
['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Item Discount Rate', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Region', 'Order State', 'Product Card Id', 'Product Name', 'Product Price', 'Shipping Mode']


In [14]:
columns_to_drop_final = [
    'Order Item Profit Ratio',
    'Customer Zipcode',
    'Product Card Id',
    'Customer Id'
]

df_working = df_working.drop(columns=columns_to_drop_final)
print(f"Remaining columns: {df_working.shape[1]}")
print(df_working.columns.tolist())

Remaining columns: 23
['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Item Discount Rate', 'Order Item Quantity', 'Sales', 'Order Region', 'Order State', 'Product Name', 'Product Price', 'Shipping Mode']


## Date Adjustment

In [15]:
# Extract date features
df_working['order_month'] = df_working['order date (DateOrders)'].dt.month
df_working['order_dayofweek'] = df_working['order date (DateOrders)'].dt.dayofweek
df_working['order_quarter'] = df_working['order date (DateOrders)'].dt.quarter
df_working['order_year'] = df_working['order date (DateOrders)'].dt.year
df_working['order_hour'] = df_working['order date (DateOrders)'].dt.hour

# Drop original date column
df_working = df_working.drop(columns=['order date (DateOrders)'])

print(df_working.shape)
print(df_working.columns.tolist())

(162467, 27)
['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Item Discount Rate', 'Order Item Quantity', 'Sales', 'Order Region', 'Order State', 'Product Name', 'Product Price', 'Shipping Mode', 'order_month', 'order_dayofweek', 'order_quarter', 'order_year', 'order_hour']


In [16]:
df_working[['order_month', 'order_dayofweek', 'order_quarter', 'order_year', 'order_hour']].describe()

,order_month,order_dayofweek,order_quarter,order_year,order_hour
count,162467.000000,162467.000000,162467.000000,162467.000000,162467.000000
mean,5.980919,3.010716,2.339109,2015.843765,11.482547
std,3.347177,2.000671,1.088766,0.768501,6.925032
min,1.000000,0.000000,1.000000,2015.000000,0.000000
25%,3.000000,1.000000,1.000000,2015.000000,5.000000
50%,6.000000,3.000000,2.000000,2016.000000,11.000000
75%,9.000000,5.000000,3.000000,2016.000000,17.000000
max,12.000000,6.000000,4.000000,2017.000000,23.000000


In [18]:
df_working.dtypes

Type                                 str
Days for shipment (scheduled)      int64
Late_delivery_risk                 int64
Category Name                        str
Customer City                        str
Customer Country                     str
Customer Segment                     str
Customer State                       str
Department Name                      str
Latitude                         float64
Longitude                        float64
Market                               str
Order City                           str
Order Country                        str
Order Item Discount Rate         float64
Order Item Quantity                int64
Sales                            float64
Order Region                         str
Order State                          str
Product Name                         str
Product Price                    float64
Shipping Mode                        str
order_month                        int32
order_dayofweek                    int32
order_quarter   

In [20]:
categorical_cols = df_working.select_dtypes(include='object').columns.tolist()
print("Categorical columns and unique value counts:\n")
for col in categorical_cols:
    print(f"{col}: {df_working[col].nunique()} unique values")

Categorical columns and unique value counts:

Type: 4 unique values
Category Name: 31 unique values
Customer City: 562 unique values
Customer Country: 2 unique values
Customer Segment: 3 unique values
Customer State: 44 unique values
Department Name: 6 unique values
Market: 5 unique values
Order City: 3576 unique values
Order Country: 164 unique values
Order Region: 23 unique values
Order State: 1089 unique values
Product Name: 87 unique values
Shipping Mode: 4 unique values


C:\Users\ferna\AppData\Local\Temp\ipykernel_26156\1310896751.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_working.select_dtypes(include='object').columns.tolist()


## Encoding

In [21]:
from sklearn.preprocessing import LabelEncoder

# Drop high cardinality geographic columns
df_working = df_working.drop(columns=[
    'Order City', 'Order State', 'Customer City',
    'Customer State', 'Order Country', 'Customer Country'
])

# One-Hot Encode low cardinality columns
ohe_cols = ['Type', 'Customer Segment', 'Department Name', 'Market', 'Shipping Mode']
df_working = pd.get_dummies(df_working, columns=ohe_cols, drop_first=False)

# Label Encode medium cardinality columns
le = LabelEncoder()
for col in ['Category Name', 'Order Region', 'Product Name']:
    df_working[col] = le.fit_transform(df_working[col])

print(f"Final shape: {df_working.shape}")
print(df_working.dtypes)

Final shape: (162467, 38)
Days for shipment (scheduled)      int64
Late_delivery_risk                 int64
Category Name                      int64
Latitude                         float64
Longitude                        float64
Order Item Discount Rate         float64
Order Item Quantity                int64
Sales                            float64
Order Region                       int64
Product Name                       int64
Product Price                    float64
order_month                        int32
order_dayofweek                    int32
order_quarter                      int32
order_year                         int32
order_hour                         int32
Type_CASH                           bool
Type_DEBIT                          bool
Type_PAYMENT                        bool
Type_TRANSFER                       bool
Customer Segment_Consumer           bool
Customer Segment_Corporate          bool
Customer Segment_Home Office        bool
Department Name_Apparel        

In [22]:
# Convert bool columns to int
bool_cols = df_working.select_dtypes(include='bool').columns.tolist()
df_working[bool_cols] = df_working[bool_cols].astype(int)

# Final verification
print(f"Shape: {df_working.shape}")
print(f"\nAny remaining non-numeric columns: {df_working.select_dtypes(include='object').columns.tolist()}")
print(f"\nAny missing values: {df_working.isnull().sum().sum()}")
print(f"\nTarget distribution:\n{df_working['Late_delivery_risk'].value_counts(normalize=True) * 100}")

Shape: (162467, 38)

Any remaining non-numeric columns: []

Any missing values: 0

Target distribution:
Late_delivery_risk
1    54.787126
0    45.212874
Name: proportion, dtype: float64


## Save dataset

In [23]:
df_working.to_csv('../data/df_working_clean.csv', index=False)